## SNN Model

This is the final SNN model used as described in the thesis.

In [832]:
import urllib.request
import zipfile
from pathlib import Path

url = "https://ndownloader.figshare.com/files/38671817"
out = Path("bcg_dataset.zip")

urllib.request.urlretrieve(url, out)
print(f"Downloaded {out.stat().st_size / 1e6:.1f} MB to {out.resolve()}")

with zipfile.ZipFile(out) as z:
    z.extractall(".")  # uses whatever folder structure the zip already has
print("Extracted")

Downloaded 52.1 MB to /Users/mazlu/Desktop/VSCode/Bachelorthesis/ppg-bp-snn/bcg_dataset.zip
Extracted


In [833]:
import torch
import torch.nn as nn
import torch.optim as optim
import snntorch as snn
from snntorch import spikegen
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from snntorch import surrogate, functional as SF
from scipy.stats import beta as beta_dist
import pickle
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
import scipy.io as sio
import os
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [834]:
import pickle
import numpy as np
from pathlib import Path

with open("FeatureRanking_Results/rf_results.pkl", "rb") as f:
    res = pickle.load(f)

# 1. All keys
print("ALL KEYS:")
print("=" * 60)
for key in sorted(res.keys()):
    val = res[key]
    if isinstance(val, list) and len(val) > 0 and isinstance(val[0], tuple):
        print(f"  {key:50s} → {len(val)} features")
    else:
        print(f"  {key:50s} → {val}")

# 2. Final feature sets (after full pipeline) for each mode
modes = ["impurity_indep", "impurity_dep", "dropcol_indep", "dropcol_dep"]

print("\n\nFINAL FEATURES PER MODE (after Spearman filtering):")
print("=" * 60)
for mode in modes:
    sbp_key = f"sbp_ranking_spearman_{mode}"
    dbp_key = f"dbp_ranking_spearman_{mode}"
    if sbp_key not in res:
        print(f"\n  {mode}: NOT YET RUN")
        continue
    sbp = [f for f, _ in res[sbp_key]]
    dbp = [f for f, _ in res[dbp_key]]
    combined = list(dict.fromkeys(sbp[:10] + dbp[:10]))
    print(f"\n  {mode}:")
    print(f"    SBP features remaining: {len(sbp)}")
    print(f"    DBP features remaining: {len(dbp)}")
    print(f"    SBP top 10: {sbp[:10]}")
    print(f"    DBP top 10: {dbp[:10]}")
    print(f"    Combined unique (top 10 each): {len(combined)} → {combined}")

# 3. Cross-mode agreement
print("\n\nCROSS-MODE AGREEMENT (top 10 SBP after Spearman):")
print("=" * 60)
all_sbp_sets = {}
for mode in modes:
    key = f"sbp_ranking_spearman_{mode}"
    if key in res:
        all_sbp_sets[mode] = set(f for f, _ in res[key][:10])

if len(all_sbp_sets) == 4:
    shared = set.intersection(*all_sbp_sets.values())
    print(f"  Features in ALL 4 modes: {len(shared)} → {sorted(shared)}")
    for mode, feats in all_sbp_sets.items():
        unique = feats - set.union(*[v for k, v in all_sbp_sets.items() if k != mode])
        if unique:
            print(f"  Only in {mode}: {sorted(unique)}")

print("\n\nCROSS-MODE AGREEMENT (top 10 DBP after Spearman):")
print("=" * 60)
all_dbp_sets = {}
for mode in modes:
    key = f"dbp_ranking_spearman_{mode}"
    if key in res:
        all_dbp_sets[mode] = set(f for f, _ in res[key][:10])

if len(all_dbp_sets) == 4:
    shared = set.intersection(*all_dbp_sets.values())
    print(f"  Features in ALL 4 modes: {len(shared)} → {sorted(shared)}")
    for mode, feats in all_dbp_sets.items():
        unique = feats - set.union(*[v for k, v in all_dbp_sets.items() if k != mode])
        if unique:
            print(f"  Only in {mode}: {sorted(unique)}")

# 4. Pipeline reduction summary
print("\n\nPIPELINE REDUCTION SUMMARY:")
print("=" * 60)
print(f"  {'Mode':<25} {'Start':>6} {'→ Pearson':>10} {'→ Spearman':>12}")
print(f"  {'-'*25} {'-'*6} {'-'*10} {'-'*12}")
for mode in modes:
    p_key = f"sbp_ranking_pearson_{mode}"
    s_key = f"sbp_ranking_spearman_{mode}"
    if p_key in res and s_key in res:
        n_pearson = len(res[p_key])
        n_spearman = len(res[s_key])
        print(f"  {mode:<25} {'218':>6} {'→ ' + str(n_pearson):>10} {'→ ' + str(n_spearman):>12}")

ALL KEYS:
  dbp_ranking                                        → 218 features
  dbp_ranking_pearson_dropcol_dep                    → 124 features
  dbp_ranking_pearson_dropcol_indep                  → 124 features
  dbp_ranking_pearson_impurity_dep                   → 61 features
  dbp_ranking_pearson_impurity_indep                 → 60 features
  dbp_ranking_spearman_dropcol_dep                   → 30 features
  dbp_ranking_spearman_dropcol_indep                 → 30 features
  dbp_ranking_spearman_impurity_dep                  → 18 features
  dbp_ranking_spearman_impurity_indep                → 18 features
  dbp_ranking_subject_dependent                      → 218 features
  dropcol_dbp_ranking                                → 218 features
  dropcol_dbp_ranking_subject_dependent              → 218 features
  dropcol_folds                                      → 5
  dropcol_n_estimator                                → 50
  dropcol_sbp_ranking                                → 218 featur

In [835]:
num_steps = 164
THRESHOLD_HIDDEN = 0.3
THRESHOLD_OUTPUT = 0.1
SCALING_FACTOR = 0.1
SPIKE_GAIN = 1.3
SMOOTHING = True
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

num_folds = 5
lr = 5e-4
eps = 1e-8

kernel_tau = 20.0
kernel_decay = torch.exp(torch.tensor(-1.0 / kernel_tau))

# TO FIX: the abp signal is not following the original signal. Something with the loss is fucked up.
# if there are 2 spikes in the original signal, only the first one is detected. This is pure guessing at this point

LAMBDA_LM = 0.5
SP_MEAN_ALL = 121.0
DP_MEAN_ALL = 67.2

CONN_PROB = 0.5

## Data Loading


In [836]:
def load_data(fold_num):
    return pd.read_csv(f'bcg_dataset/feat_fold_{fold_num}.csv')

def load_abp(fold_num):
    mat = sio.loadmat(f'bcg_dataset/signal_mabp_fold_{fold_num}.mat')
    return mat['abp_signal']

def load_limits(fold_num):
    mat = sio.loadmat(f'bcg_dataset/signal_mabp_fold_{fold_num}.mat')
    return mat['class_limits'] - 1

abp_all = np.concatenate([load_abp(i) for i in range(num_folds)])

# Computed from all training data, stored as a fixed constant.
GLOBAL_MEAN_T0 = (abp_all[:, 0] / 100.0).mean()
print(f'Global mean first ABP timestep (normalised): {GLOBAL_MEAN_T0:.4f}')
print(f'= {GLOBAL_MEAN_T0 * 100:.2f} mmHg')

Global mean first ABP timestep (normalised): 0.6725
= 67.25 mmHg


## Plotting

In [837]:
def plot_predictions(all_true, all_predictions, label, fold, samplingRange):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].scatter(all_true, all_predictions, alpha=0.4, s=10)
    axes[0].plot([all_true.min(), all_true.max()],
                 [all_true.min(), all_true.max()], 'r--', label='Perfect prediction')
    axes[0].set_xlabel(f'True {label}'); axes[0].set_ylabel(f'Predicted {label}')
    axes[0].set_title(f'Fold {fold+1} - Predicted vs True {label}');  axes[0].legend()
    axes[1].plot(all_true[:samplingRange], label=f'True {label}', alpha=0.8)
    axes[1].plot(all_predictions[:samplingRange], label=f'Predicted {label}', alpha=0.8)
    axes[1].set_xlabel('Sample'); axes[1].set_ylabel(f'{label} (mmHg)')
    axes[1].set_title(f'Fold {fold+1} - First {samplingRange} Samples');  axes[1].legend()
    plt.tight_layout()
    
    if label=='SBP':
        plt.savefig(f'ScatterPlot_impurity_indep_False/scatter_sbp_fold_{fold+1}.png')
    else:
        plt.savefig(f'ScatterPlot_impurity_indep_False/scatter_dbp_fold_{fold+1}.png')
    

    plt.close()


def plot_signal(signal, spk_rec, abp_target, lim_batch, sample_idx=0):
    sig = signal[:, sample_idx, 0].detach().cpu().numpy()
    spk = spk_rec[:, sample_idx, 0].detach().cpu().numpy()
    target = (abp_target[sample_idx] / 100.0).detach().cpu().numpy()
    cycle_end = int(lim_batch[sample_idx, -1].item())
    spike_times = np.where(spk > 0)[0]

    plt.close('all')
    plt.figure(figsize=(10, 4))
    for t in spike_times:
        plt.axvline(x=t, color='black', alpha=0.4, linewidth=0.8)
    plt.plot(sig, color='steelblue', label='Kernel output signal', zorder=2)
    plt.plot(target, color='r', linestyle='--', label='True ABP (visual reference only)')
    plt.axvline(x=cycle_end, color='green', linestyle='-', linewidth=2,
                label=f'First cycle end (t={cycle_end})', zorder=3)
    plt.xlabel('Timestep'); plt.ylabel('BP (normalised)')
    plt.title(f'Sample {sample_idx} - Kernel output vs ABP target')
    plt.legend(); plt.show()

## Core Functions

In [838]:
def extract_BP_limits(sig, limits_batch):
    s = sig[:, :, 0]                                    # [T, B]
    B = s.shape[1]
    cycle_end = limits_batch[:, -1]                      # end of first cycle

    pred_sbp = torch.stack([s[:cycle_end[b], b].max() for b in range(B)]) * 100.0

    peak_idx = torch.stack([s[:cycle_end[b], b].argmax() for b in range(B)])

    pred_dbp = torch.stack([s[peak_idx[b]:cycle_end[b], b].min() for b in range(B)]) * 100.0

    return pred_sbp, pred_dbp


def apply_exponential_kernel(spk_rec, decay, abp_t0=None):
    """
    abp_t0 : [B, 1] — TRAINING ONLY, per-sample init from true first ABP timestep.
              None at eval → falls back to GLOBAL_MEAN_T0.
    """
    if abp_t0 is not None:
        signal = (abp_t0 / SCALING_FACTOR).expand_as(spk_rec[0]).clone()
    else:
        signal = torch.ones_like(spk_rec[0]) * (GLOBAL_MEAN_T0 / SCALING_FACTOR)
    out = []
    for t in range(spk_rec.shape[0]):
        signal = decay * signal + spk_rec[t] * SPIKE_GAIN
        out.append(signal)
    return torch.stack(out, dim=0)


def smooth(sig, window=10):
    kernel = torch.ones(1, 1, window, device=sig.device) / window
    x = sig.permute(1, 2, 0)
    x = torch.nn.functional.pad(x, (window - 1, 0))
    x = torch.nn.functional.conv1d(x, kernel, groups=1)
    return x.permute(2, 0, 1)



'''
def weighted_landmark_loss(sig, target, lim, bp_true, bp_mean):
    B = sig.shape[1]
    weight = 1.0 + torch.abs(bp_true.cpu() - bp_mean) / bp_mean
    weight = weight.to(sig.device)

    waveform_mae = torch.mean(torch.abs(sig - target), dim=0)[:, 0]
    waveform_loss = (waveform_mae * weight).mean()

    sys_idx = lim[:, 0]
    dia_idx = lim[:, 2]
    lm_errors = torch.stack([
        torch.abs(sig[sys_idx[b], b, 0] - target[sys_idx[b], b, 0]) + torch.abs(sig[dia_idx[b], b, 0] - target[dia_idx[b], b, 0])
        for b in range(B)
    ])
    landmark_loss = (lm_errors * weight).mean()

    return waveform_loss + LAMBDA_LM * landmark_loss
'''

def weighted_landmark_loss(sig, target, lim, bp_true, bp_mean):
    B = sig.shape[1]
    T = sig.shape[0]
    weight = 1.0 + torch.abs(bp_true.cpu() - bp_mean) / bp_mean
    weight = weight.to(sig.device)

    # mask: only compute waveform loss over first cycle
    idx = torch.arange(T, device=sig.device).view(T, 1)
    cycle_end = lim[:, -1].view(1, B).to(sig.device)
    mask = (idx < cycle_end).float().unsqueeze(-1)          # [T, B, 1]

    abs_err = torch.abs(sig - target) * mask
    waveform_mae = abs_err.sum(dim=0)[:, 0] / mask.sum(dim=0)[:, 0].clamp(min=1)
    waveform_loss = (waveform_mae * weight).mean()

    sys_idx = lim[:, 0]
    dia_idx = lim[:, 2]
    lm_errors = torch.stack([
        torch.abs(sig[sys_idx[b], b, 0] - target[sys_idx[b], b, 0]) + torch.abs(sig[dia_idx[b], b, 0] - target[dia_idx[b], b, 0])
        for b in range(B)
    ])
    landmark_loss = (lm_errors * weight).mean()

    return waveform_loss + LAMBDA_LM * landmark_loss

## Network

In [839]:
def make_spike_time_encoding(num_steps, num_timers, device):
    anchor_times = [17, 51, 115, 130]
    num_remaining = num_timers - len(anchor_times)
    uniform_times = torch.linspace(0, num_steps - 1, num_remaining).long().tolist()
    all_times = sorted(set(anchor_times + uniform_times))[:num_timers]
    enc = torch.zeros(num_steps, num_timers, device=device)
    for i, t in enumerate(all_times):
        enc[t, i] = 1.0
    return enc

class Net(nn.Module):
    def __init__(self, num_inputs, num_hidden, num_outputs, beta_val, num_steps=164, num_timers=10):
        super().__init__()
        self.num_steps = num_steps
        self.num_timers = num_timers

        beta1 = torch.clamp(torch.randn(num_hidden) * 0.05 + beta_val, 0.5, 0.99)
        beta2 = torch.clamp(torch.randn(num_outputs) * 0.05 + beta_val, 0.5, 0.99)

        self.fc1 = nn.Linear(num_inputs + num_timers, num_hidden)
        self.lif1 = snn.Leaky(beta=beta1, learn_beta=True, threshold=THRESHOLD_HIDDEN)

        self.fc2 = nn.Linear(num_hidden, num_outputs)
        self.lif2 = snn.Leaky(beta=beta2, learn_beta=True, threshold=THRESHOLD_OUTPUT)

        mask1 = (torch.rand_like(self.fc1.weight) < CONN_PROB).float()
        mask1[:, -num_timers:] = 1.0  # timer neurons fully connected to all hidden neurons
        self.register_buffer("mask1", mask1)
        self.register_buffer("mask2", (torch.rand_like(self.fc2.weight) < CONN_PROB).float())

    def forward(self, xfeatures):
        mem1 = self.lif1.reset_mem()
        mem2 = self.lif2.reset_mem()

        spk_rec = []

        input_spikes = spikegen.rate(xfeatures, num_steps=self.num_steps).to(device)
        time_enc = make_spike_time_encoding(self.num_steps, self.num_timers, device=xfeatures.device)

        for t in range(self.num_steps):
            t_spk = time_enc[t].unsqueeze(0).expand(input_spikes[t].shape[0], -1)
            inp = torch.cat([input_spikes[t], t_spk], dim=-1)

            cur1 = nn.functional.linear(inp, self.fc1.weight * self.mask1, self.fc1.bias)
            spk1, mem1 = self.lif1(cur1, mem1)

            cur2 = nn.functional.linear(spk1, self.fc2.weight * self.mask2, self.fc2.bias)
            spk2, mem2 = self.lif2(cur2, mem2)

            spk_rec.append(spk2)

        return torch.stack(spk_rec, dim=0).mean(dim=-1, keepdim=True)

## Load Features

In [840]:
with open('FeatureRanking_Results/rf_results.pkl', 'rb') as f:
    ranking = pickle.load(f)

modes = ["impurity_indep", "impurity_dep", "dropcol_indep", "dropcol_dep"]

for mode in modes:
    sbp_key = f"sbp_ranking_spearman_{mode}"
    dbp_key = f"dbp_ranking_spearman_{mode}"
    
    if sbp_key not in ranking:
        print(f"\n{mode}: NOT FOUND")
        continue
    
    sbp = [f for f, _ in ranking[sbp_key][:10]]
    dbp = [f for f, _ in ranking[dbp_key][:10]]
    combined = list(dict.fromkeys(sbp + dbp))
    
    print(f"\n{'='*50}")
    print(f"  {mode}")
    print(f"{'='*50}")
    print(f"  SBP top 10: {sbp}")
    print(f"  DBP top 10: {dbp}")
    print(f"  Combined:   {len(combined)} features → {combined}")


  impurity_indep
  SBP top 10: ['T_peak_e_norm', 'T_peak_e', 'ppg4_histogram_down_8', 'dsdc_12', 'ppg_b', 'vpg_histogram_down_0', 'ppg4_min_0', 'ppg4_max_0', 'usdc_2', 'vpg_histogram_down_3']
  DBP top 10: ['T_peak_e', 'ppg_histogram_down_4', 'ppg_histogram_down_5', 'ppg4_histogram_down_3', 'ppg_histogram_down_2', 'dsdc_8', 'S4', 'vpg_histogram_down_0', 'T_peak_e_norm', 'ppg_mean_cycles_match_peak_59']
  Combined:   17 features → ['T_peak_e_norm', 'T_peak_e', 'ppg4_histogram_down_8', 'dsdc_12', 'ppg_b', 'vpg_histogram_down_0', 'ppg4_min_0', 'ppg4_max_0', 'usdc_2', 'vpg_histogram_down_3', 'ppg_histogram_down_4', 'ppg_histogram_down_5', 'ppg4_histogram_down_3', 'ppg_histogram_down_2', 'dsdc_8', 'S4', 'ppg_mean_cycles_match_peak_59']

  impurity_dep
  SBP top 10: ['T_peak_e_norm', 'T_peak_e', 'ppg4_histogram_down_8', 'dsdc_12', 'ppg_b', 'vpg_histogram_down_0', 'ppg4_min_0', 'ppg4_max_0', 'usdc_2', 'vpg_histogram_down_3']
  DBP top 10: ['T_peak_e', 'ppg_histogram_down_4', 'ppg_histogram_d

In [841]:
with open('FeatureRanking_Results/rf_results.pkl', 'rb') as f:
    ranking = pickle.load(f)


with open('FeatureRanking_Results/rf_results.pkl', 'rb') as f:
    res = pickle.load(f)

for key in sorted(res.keys()):
    print(key)

RANKING_MODE = "impurity_indep" # impurity_indep, impurity_dep, dropcol_indep, dropcol_dep
USE_PIPELINE = False             # True = after filtering, False = raw ranking

if USE_PIPELINE:
    sbp_key = f"sbp_ranking_spearman_{RANKING_MODE}"
    dbp_key = f"dbp_ranking_spearman_{RANKING_MODE}"
    label = f"{RANKING_MODE} (filtered)"
else:
    RAW_KEY_MAP = {
        "impurity_indep": ("sbp_ranking", "dbp_ranking"),
        "impurity_dep": ("sbp_ranking_subject_dependent", "dbp_ranking_subject_dependent"),
        "dropcol_indep": ("dropcol_sbp_ranking", "dropcol_dbp_ranking"),
        "dropcol_dep": ("dropcol_sbp_ranking_subject_dependent", "dropcol_dbp_ranking_subject_dependent"),
    }
    sbp_key, dbp_key = RAW_KEY_MAP[RANKING_MODE]
    label = f"{RANKING_MODE} (raw)"

top_features_SBP = [f for f, _ in ranking[sbp_key][:10]]
top_features_DBP = [f for f, _ in ranking[dbp_key][:10]]
top_features_ALL = list(dict.fromkeys(top_features_SBP + top_features_DBP))

print(f'Mode: {label}')
print(f'SBP features: {top_features_SBP}')
print(f'DBP features: {top_features_DBP}')
print(f'Combined: {len(top_features_ALL)} features')

dbp_ranking
dbp_ranking_pearson_dropcol_dep
dbp_ranking_pearson_dropcol_indep
dbp_ranking_pearson_impurity_dep
dbp_ranking_pearson_impurity_indep
dbp_ranking_spearman_dropcol_dep
dbp_ranking_spearman_dropcol_indep
dbp_ranking_spearman_impurity_dep
dbp_ranking_spearman_impurity_indep
dbp_ranking_subject_dependent
dropcol_dbp_ranking
dropcol_dbp_ranking_subject_dependent
dropcol_folds
dropcol_n_estimator
dropcol_sbp_ranking
dropcol_sbp_ranking_subject_dependent
dropcol_seeds
folds
n_estimator
sbp_ranking
sbp_ranking_pearson_dropcol_dep
sbp_ranking_pearson_dropcol_indep
sbp_ranking_pearson_impurity_dep
sbp_ranking_pearson_impurity_indep
sbp_ranking_spearman_dropcol_dep
sbp_ranking_spearman_dropcol_indep
sbp_ranking_spearman_impurity_dep
sbp_ranking_spearman_impurity_indep
sbp_ranking_subject_dependent
seeds
Mode: impurity_indep (raw)
SBP features: ['TdiaToEnd', 'AUCsys_norm', 'AUCdia_norm', 'T_peak_e_norm', 'ppg_histogram_down_2', 'T_peak_a', 'SteepDiaRise', 'TNegSteepest_norm', 'vpg_max_

## Training Loop

In [842]:
set_seed()
batch_size = 32
num_epochs = 15
num_inputs = len(top_features_ALL)
num_hidden = 32
num_outputs = 10
num_timer_neurons = 10
beta_val = 0.9

cv_mae_sbp = 0.0
cv_mae_dbp = 0.0
cv_total_samples = 0
fold_maes_sbp = []
fold_maes_dbp = []
fold_mases_sbp = []
fold_mases_dbp = []

os.makedirs('checkpoints', exist_ok=True)

for fold in range(num_folds):
    print(f'\nFold {fold+1}/{num_folds}')

    net = Net(num_inputs, num_hidden, num_outputs, beta_val, num_steps, num_timer_neurons)
    optimizer = optim.Adam(net.parameters(), lr=lr)

    test_data = load_data(fold)
    train_data = pd.concat([load_data(i) for i in range(num_folds) if i != fold])

    abp_test = torch.tensor(load_abp(fold), dtype=torch.float32)
    abp_train = torch.cat([
        torch.tensor(load_abp(i), dtype=torch.float32)
        for i in range(num_folds) if i != fold
    ], dim=0)

    lim_test = torch.tensor(load_limits(fold), dtype=torch.long)
    lim_train = torch.cat([
        torch.tensor(load_limits(i), dtype=torch.long)
        for i in range(num_folds) if i != fold
    ], dim=0)

    X_train = torch.tensor(train_data[top_features_ALL].values, dtype=torch.float32)
    X_test = torch.tensor(test_data[top_features_ALL].values,  dtype=torch.float32)
    y_train = torch.tensor(train_data.iloc[:, 2:4].values, dtype=torch.float32)
    y_test = torch.tensor(test_data.iloc[:, 2:4].values,  dtype=torch.float32)

    mn = X_train.min(0).values;  mx = X_train.max(0).values
    X_train = (X_train - mn) / (mx - mn + eps)
    X_test = (X_test  - mn) / (mx - mn + eps)

    train_loader = DataLoader(
        TensorDataset(X_train, y_train, abp_train, lim_train),
        batch_size=batch_size, shuffle=True
    )
    # abp_test in test loader for plot_signal only — not used in prediction
    test_loader = DataLoader(
        TensorDataset(X_test, y_test, abp_test, lim_test),
        batch_size=batch_size, shuffle=False
    )

    # ── Training ──────────────────────────────────────────────
    for epoch in range(num_epochs):
        net.train()
        num_batches = total_loss = 0

        for x, y, abp, lim in train_loader:
            x = x.to(device)
            y, abp = y.to(device), abp.to(device)
            target = (abp / 100.0).permute(1, 0).unsqueeze(-1)   # [T, B, 1]
            abp_t0 = (abp[:, 0:1] / 100.0)                      # [B, 1]

            optimizer.zero_grad()
            spk = net(x)
            sig = apply_exponential_kernel(spk, kernel_decay, abp_t0) * SCALING_FACTOR
            if SMOOTHING:
                sig = smooth(sig)
            loss = weighted_landmark_loss(sig, target, lim, y[:, 0], SP_MEAN_ALL)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
            optimizer.step()

            with torch.no_grad():
                net.lif1.beta.clamp_(0.5, 0.99)
                net.lif2.beta.clamp_(0.5, 0.99)

            total_loss += loss.item()
            num_batches += 1

        #print(f'    Epoch {epoch+1:2d} | Loss: {total_loss/num_batches:.4f}')

    #torch.save(net.state_dict(), f'checkpoints/checkpoint_fold_{fold+1}.pt')
    #print(f'    Saved checkpoint: checkpoints/checkpoint_fold_{fold+1}.pt')

    # ── Evaluation ────────────────────────────────────────────
    net.eval()
    total_mae_sbp = total_mae_dbp = total_samples = 0
    true_sbp_list = [];  pred_sbp_list = []
    true_dbp_list = [];  pred_dbp_list = []
    first_batch = True

    with torch.no_grad():
        for x, y, abp, lim in test_loader:
            x = x.to(device)
            y = y.to(device)

            spk = net(x)
            # abp_t0=None → kernel starts from GLOBAL_MEAN_T0
            sig = apply_exponential_kernel(spk, kernel_decay) * SCALING_FACTOR
            if SMOOTHING:
                sig = smooth(sig)

            pred_sbp, pred_dbp = extract_BP_limits(sig, lim)
            '''
            if first_batch:
                plot_signal(sig, spk, abp, lim, sample_idx=0)
                first_batch = False
            '''

            total_mae_sbp += torch.abs(pred_sbp - y[:, 0]).sum().item()
            total_mae_dbp += torch.abs(pred_dbp - y[:, 1]).sum().item()
            total_samples += y.shape[0]

            true_sbp_list.extend(y[:, 0].cpu().numpy())
            pred_sbp_list.extend(pred_sbp.cpu().numpy())
            true_dbp_list.extend(y[:, 1].cpu().numpy())
            pred_dbp_list.extend(pred_dbp.cpu().numpy())

    fold_mae_sbp = total_mae_sbp / total_samples
    fold_mae_dbp = total_mae_dbp / total_samples
    fold_maes_sbp.append(fold_mae_sbp)
    fold_maes_dbp.append(fold_mae_dbp)
    cv_mae_sbp += total_mae_sbp
    cv_mae_dbp += total_mae_dbp
    cv_total_samples += total_samples

    print(f'    → Fold {fold+1} MAE  SBP: {fold_mae_sbp:.4f} DBP: {fold_mae_dbp:.4f}')

    # Naive baseline: always predict the training mean
    #naive_mae_sbp = np.mean(np.abs(y_train[:, 0].numpy() - y_train[:, 0].mean().item()))
    #naive_mae_dbp = np.mean(np.abs(y_train[:, 1].numpy() - y_train[:, 1].mean().item()))
    train_mean_sbp = y_train[:, 0].mean().item()
    train_mean_dbp = y_train[:, 1].mean().item()
    naive_mae_sbp = np.mean(np.abs(y_test[:, 0].numpy() - train_mean_sbp))
    naive_mae_dbp = np.mean(np.abs(y_test[:, 1].numpy() - train_mean_dbp))

    fold_mase_sbp = fold_mae_sbp / naive_mae_sbp * 100
    fold_mase_dbp = fold_mae_dbp / naive_mae_dbp * 100

    print(f'    → MASE  SBP: {fold_mase_sbp:.1f}%  DBP: {fold_mase_dbp:.1f}%')
    fold_mases_sbp.append(fold_mase_sbp)
    fold_mases_dbp.append(fold_mase_dbp)

    plot_predictions(np.array(true_sbp_list), np.array(pred_sbp_list), 'SBP', fold, 500)
    plot_predictions(np.array(true_dbp_list), np.array(pred_dbp_list), 'DBP', fold, 500)

print(f"\nFinal CV MAE's:")
print(f" SBP: {cv_mae_sbp/cv_total_samples:.4f} ± {np.std(fold_maes_sbp):.4f}")
print(f" DBP: {cv_mae_dbp/cv_total_samples:.4f} ± {np.std(fold_maes_dbp):.4f}")

# Global MASE: compute the MASE like the benchmark paper
all_naive_sbp = []
all_naive_dbp = []
for f in range(num_folds):
    tr = pd.concat([load_data(i) for i in range(num_folds) if i != f])
    te = load_data(f)
    tr_mean_sbp = tr.iloc[:, 2].mean()
    tr_mean_dbp = tr.iloc[:, 3].mean()
    all_naive_sbp.extend(np.abs(te.iloc[:, 2].values - tr_mean_sbp))
    all_naive_dbp.extend(np.abs(te.iloc[:, 3].values - tr_mean_dbp))

global_naive_sbp = np.mean(all_naive_sbp)
global_naive_dbp = np.mean(all_naive_dbp)
global_mase_sbp = (cv_mae_sbp / cv_total_samples) / global_naive_sbp * 100
global_mase_dbp = (cv_mae_dbp / cv_total_samples) / global_naive_dbp * 100
print(f"\n SBP MASE: {global_mase_sbp:.1f}%")
print(f" DBP MASE: {global_mase_dbp:.1f}%")


Fold 1/5
    → Fold 1 MAE  SBP: 17.3764 DBP: 9.7294
    → MASE  SBP: 100.6%  DBP: 124.5%

Fold 2/5
    → Fold 2 MAE  SBP: 13.8283 DBP: 8.5999
    → MASE  SBP: 119.7%  DBP: 107.3%

Fold 3/5
    → Fold 3 MAE  SBP: 10.2245 DBP: 6.0234
    → MASE  SBP: 132.5%  DBP: 94.5%

Fold 4/5
    → Fold 4 MAE  SBP: 10.7978 DBP: 7.5061
    → MASE  SBP: 94.1%  DBP: 83.5%

Fold 5/5
    → Fold 5 MAE  SBP: 12.0387 DBP: 7.5445
    → MASE  SBP: 118.0%  DBP: 103.3%

Final CV MAE's:
 SBP: 12.9549 ± 2.5772
 DBP: 7.9039 ± 1.2360

 SBP MASE: 110.5%
 DBP MASE: 103.1%
